Notebook to compare different pairing functions. 

In [1]:
import sys
import pandas as pd
import numpy as np
import os
sys.path.append(os.path.abspath("../0_UTILITY_FUNCTIONS/"))

from get_LrLx_data import pair_obs_alg
from get_data import*

## Alternative Pairing Algorithms

In [2]:
def pair_obs_closest(radio_df, xray_df, dt_mjd=1, verbose=True):
    """
    For every radio observation, find the single closest X-ray observation within dt_mjd of it.
    - If there are X-ray detections (Fx_uplim_bool == False) in the window, choose the closest detection.
    - Otherwise, if only upper limits exist in the window, choose the closest upper limit.
    - If no X-ray points lie in the window, the radio observation is unpaired.
    Returns (paired_data_df, unpaired_radio_dates_array).
    """
    source_name = radio_df["name"].to_numpy()[0]

    xray_MJDs = xray_df["t_xray"].to_numpy()  # assumed sorted low->high
    Fx = xray_df["Fx"].to_numpy()
    Fx_unc_l = xray_df["Fx_unc_l"].to_numpy()
    Fx_unc_u = xray_df["Fx_unc_u"].to_numpy()
    uplims_xray = xray_df["Fx_uplim_bool"].to_numpy()
    xray_states = xray_df["Xstate"].to_numpy()

    radio_MJDs = radio_df["t_radio"].to_numpy()  # assumed sorted low->high
    Fr = radio_df["Fr"].to_numpy()
    Fr_unc = radio_df["Fr_unc"].to_numpy()
    uplims_radio = radio_df["Fr_uplim_bool"].to_numpy()
    radio_states = radio_df["Rstate"].to_numpy()

    paired_data = []
    unpaired_radio_dates = []

    if verbose:
        print('{:<20s}{:<20s}{:<20s}{:<10s}{:<20s}{:<20s}{:<20s}{:<20s}{:<20s}{:<15s}{:<15s}{:<15s}'.format(
            "t_radio", "Fr [mJy]", "Fr_unc [mJy]", "nx_in_bin", "t_xray_closest", "t_diff", "Fx [erg/cm^2/s]", "Fx_unc_l", "Fx_unc_u", "Fr_uplim_bool", "Fx_uplim_bool", "state"))

    for i, t in enumerate(radio_MJDs):
        # mask X-rays in the time window
        mask = (xray_MJDs >= (t - dt_mjd)) & (xray_MJDs < (t + dt_mjd))
        if not np.any(mask):
            # no xray in bin -> unpaired
            unpaired_radio_dates.append(t)
            continue

        # Extract candidates
        cand_idx = np.nonzero(mask)[0]  # indices in xray arrays
        cand_uplims = uplims_xray[cand_idx]
        cand_states = xray_states[cand_idx]

        # state mismatch warning
        if any(state != radio_states[i] for state in cand_states) and verbose:
            print("Warning: Some values in xray_states do not match radio_state. xray_states = {}, radio_state = {}".format(cand_states, radio_states[i]))

        # Prefer detections; if any detections, restrict candidates to detections only
        if np.any(~cand_uplims):
            det_mask = ~cand_uplims
            sel_idx_candidates = cand_idx[det_mask]  # indices in original xray arrays
        else:
            # only upper limits present
            sel_idx_candidates = cand_idx  # keep all (they are all uplims)

        # compute distances to radio time, choose the candidate with minimum abs diff
        sel_times = xray_MJDs[sel_idx_candidates]
        time_diffs = np.abs(sel_times - t)
        minpos = np.argmin(time_diffs)
        chosen_idx = sel_idx_candidates[minpos]

        # gather chosen xray values
        chosen_time = xray_MJDs[chosen_idx]
        chosen_Fx = Fx[chosen_idx]
        chosen_Fx_unc_l = Fx_unc_l[chosen_idx]
        chosen_Fx_unc_u = Fx_unc_u[chosen_idx]
        chosen_uplim = uplims_xray[chosen_idx]

        # compute t_diff
        t_diff = abs(chosen_time - t)

        # append to paired_data
        paired_data.append({
            "name": source_name,
            "t": t,
            "t_diff": t_diff,
            "Fr": Fr[i],
            "Fr_unc": Fr_unc[i],
            "Fr_uplim_bool": bool(uplims_radio[i]),
            "Fx": chosen_Fx,
            "Fx_unc_l": chosen_Fx_unc_l,
            "Fx_unc_u": chosen_Fx_unc_u,
            "Fx_uplim_bool": bool(chosen_uplim),
            "t_xray_chosen": chosen_time,
            "state": radio_states[i]
        })

        if verbose:
            print('{:<20.9f}{:<20.5f}{:<20.5f}{:<10d}{:<20.9f}{:<20.9f}{:<20.5e}{:<20.5e}{:<20.5e}{:<15s}{:<15s}{:<15s}'.format(
                t, Fr[i], Fr_unc[i], len(cand_idx), chosen_time, t_diff, chosen_Fx, chosen_Fx_unc_l, chosen_Fx_unc_u, str(uplims_radio[i]), str(chosen_uplim), str(radio_states[i])
            ))

    paired_data = pd.DataFrame(paired_data)
    unpaired_radio_dates = np.array(unpaired_radio_dates).reshape(-1,)

    return paired_data, unpaired_radio_dates


#################################################


## Simple pairing algorithm
# Data needs to be sorted before algorithm is run.
def pair_obs_closest_alt(radio_data, xray_data, dt=1, add_error=True):

    # Sort the data by time (should already be sorted, but just to be sure)
    radio_data = radio_data.sort_values(by="t_radio").reset_index(drop=True)
    xray_data = xray_data.sort_values(by="t_xray").reset_index(drop=True)


    xray_MJDs = xray_data["t_xray"].to_numpy()
    radio_MJDs = radio_data["t_radio"].to_numpy()

    # Initialise arrays to track pairing
    radio_paired = [False] * len(radio_MJDs)
    xray_paired = [False] * len(xray_MJDs)

    # Initialise counters
    radio_counter = 0
    xray_counter = 0
    
    # Create an empty DataFrame to store paired data
    paired_data = []
    
    # Pair observations
    while radio_counter < len(radio_MJDs) and xray_counter < len(xray_MJDs):
        
        # Get the current MJD values
        radio_MJD = radio_MJDs[radio_counter]
        xray_MJD = xray_MJDs[xray_counter]
        
        # Determine the smaller MJD
        if radio_MJD <= xray_MJD:
            MJD1, MJD2 = radio_MJD, xray_MJD
            is_radio = True  # MJD1 is from radio
        else:
            MJD1, MJD2 = xray_MJD, radio_MJD
            is_radio = False  # MJD1 is from X-ray
        
        # Check if the difference is within dt (i.e. observations should be paired)
        if MJD2 <= (MJD1 + dt): # Pair the observations

            # compute dt: signed (xray - radio) and absolute
            dt_diff = abs(xray_MJD - radio_MJD)

            # Append the combined data 
            combined_row = {
                **radio_data.iloc[radio_counter].to_dict(),
                **xray_data.iloc[xray_counter].to_dict(),  
                "dt_diff": dt_diff
            }
            paired_data.append(combined_row)
            
            # Mark as paired and advance both counters
            radio_paired[radio_counter] = True
            xray_paired[xray_counter] = True
            radio_counter += 1
            xray_counter += 1
        
        else: # No pair found, increment the counter for the smallest MJD and do not mark as paired
            if is_radio:
                radio_counter += 1
            else:
                xray_counter += 1

    # Convert paired data to DataFrame
    paired_data = pd.DataFrame(paired_data)

    #print(paired_data.loc[paired_data['t_radio'] == 58957.00347, ['Rstate', 'Fr']])
    #print()


    # For testing -- print results
    if len(paired_data)>0: print(paired_data[["t_radio", "t_xray", "dt_diff", "Fr", "Fx", "Fr_uplim_bool", "Fx_uplim_bool", "Rstate", "Xstate"]]) 


    ## Processing state definition 
    for idx, row in paired_data.iterrows():
        # Check if Rstate and Xstate are the same
        if row["Rstate"] != row["Xstate"]:
            print(
                f"Mismatch on row {idx}: "
                f"t_radio={row['t_radio']}, t_xray={row['t_xray']}, "
                f"Rstate={row['Rstate']}, Xstate={row['Xstate']}"
            )
    # Replace Xstate and Rstate columns with Rstate values
    paired_data["state"] = paired_data["Rstate"]
    # Drop the original Rstate and Xstate columns
    paired_data.drop(columns=["Rstate", "Xstate"], inplace=True)


    ## Processing time
    paired_data['t'] = (paired_data['t_radio'] + paired_data['t_xray']) / 2
    paired_data['t_diff'] = abs(paired_data['t_radio'] - paired_data['t_xray'])
    paired_data.drop(columns=['t_radio', 't_xray'], inplace=True)


    # Also get info to return the unpaired radio data
    radio_paired = np.array(radio_paired )
    radio_MJDs = np.array(radio_MJDs)

    
    # Return the DataFrame and unpaired radio MJDs
    return paired_data, radio_MJDs[~radio_paired]


#################################################

def pair_obs_fixed_bins(radio_data, xray_data, dt_mjd=1, add_error=True, weighted_ave = False):
    """
    Pairs and averages radio and X-ray observations within time bins.
    Excludes upper limits if detections exist in a bin; otherwise, averages upper limits.
    If there are multiple detections in a bin, it averages them.
    Outputs a dataframe with paired data and a list of unpaired radio observation times. 
    """

    # Sort the data by time (should already be sorted, but just to be sure)
    radio_data = radio_data.sort_values(by="t_radio").reset_index(drop=True)
    xray_data = xray_data.sort_values(by="t_xray").reset_index(drop=True)

    source_name = radio_data["name"][0]

    xray_MJDs = xray_data["t_xray"].to_numpy() # this is ordered from smallest to largest
    Fx = xray_data["Fx"].to_numpy()
    Fx_unc_l = xray_data["Fx_unc_l"].to_numpy()
    Fx_unc_u = xray_data["Fx_unc_u"].to_numpy()
    uplims_xray = xray_data["Fx_uplim_bool"].to_numpy()
    xray_states = xray_data["Xstate"].to_numpy()
    
    radio_MJDs = radio_data["t_radio"].to_numpy()  # this is ordered from smallest to largest
    Fr = radio_data["Fr"].to_numpy()
    Fr_unc = radio_data["Fr_unc"].to_numpy()
    uplims_radio = radio_data["Fr_uplim_bool"].to_numpy()
    radio_states = radio_data["Rstate"].to_numpy()
    
    start_mjd = np.min([xray_MJDs.min(), radio_MJDs.min()])
    end_mjd = np.max([xray_MJDs.max(), radio_MJDs.max()])
    interval_mjd = end_mjd - start_mjd
    nbins = int(interval_mjd / dt_mjd)
    
    # Can change the following and it would affect the pairing slightly
    current_mjd = start_mjd - 0.5 * dt_mjd # start of interval
    
    paired_data = []
    unpaired_radio_dates = []
    
    print('{:<20s}{:<20s}{:<20s}{:<10s}{:<20s}{:<10s}{:<30s}{:<15s}{:<15s}'.format(
        "Start mjd", "End mjd", "Middle mjd", "#radio", "mean radio flux", "#xray", "mean xray flux", "Radio uplim", "X-ray uplim"))
    
    for i in range(1, nbins):
        
        # Create masks for data within the current bin
        xray_mask = (xray_MJDs >= current_mjd) & (xray_MJDs < (current_mjd + dt_mjd))
        radio_mask = (radio_MJDs >= current_mjd) & (radio_MJDs < (current_mjd + dt_mjd))
        
        # Extract relevant data
        xray_MJDs_all = xray_MJDs[xray_mask]
        xray_fluxes_all = Fx[xray_mask]
        xray_fluxes_unc_l_all = Fx_unc_l[xray_mask]
        xray_fluxes_unc_u_all = Fx_unc_u[xray_mask]
        xray_uplims_all = uplims_xray[xray_mask]
        xray_states_all = xray_states[xray_mask]
        
        radio_MJDs_all = radio_MJDs[radio_mask]
        radio_fluxes_all = Fr[radio_mask]
        radio_fluxes_unc_all = Fr_unc[radio_mask]
        radio_uplims_all = uplims_radio[radio_mask]
        radio_states_all = radio_states[radio_mask]
        
        # Filter fluxes based on upper limits
        if np.any(~xray_uplims_all): # if there are any xray detections
            # Remove the upper limits
            xray_fluxes, xray_fluxes_unc_l, xray_fluxes_unc_u = xray_fluxes_all[~xray_uplims_all], xray_fluxes_unc_l_all[~xray_uplims_all] , xray_fluxes_unc_u_all[~xray_uplims_all] 
            xray_uplim = False
        else: # only upper limits
            xray_fluxes, xray_fluxes_unc_l, xray_fluxes_unc_u = xray_fluxes_all, xray_fluxes_unc_l_all , xray_fluxes_unc_u_all
            xray_uplim = True
        
        if np.any(~radio_uplims_all): # if there are any radio detections
            # Remove the upper limits
            radio_fluxes, radio_fluxes_unc = radio_fluxes_all[~radio_uplims_all], radio_fluxes_unc_all[~radio_uplims_all] 
            radio_uplim = False
        else: # only upper limits
            radio_fluxes , radio_fluxes_unc = radio_fluxes_all, radio_fluxes_unc_all
            radio_uplim = True
        
        if weighted_ave == False: # normal average
            # Apply averaging logic
            Fr_av = np.mean(radio_fluxes) if radio_fluxes.size > 0 else -5.0
            Fx_av = np.mean(xray_fluxes) if xray_fluxes.size > 0 else -5.0
            # For the averaging below, I only use propagation of uncertainties. 
            # I don't include the uncertainty due to the spread in values, as this is assumed to be much smaller.
            Fr_unc_av = np.sqrt(np.sum(radio_fluxes_unc**2)) / len(radio_fluxes) if radio_fluxes.size > 0 else -5.0
            Fx_unc_u_av = np.sqrt(np.sum(xray_fluxes_unc_u**2)) / len(xray_fluxes) if xray_fluxes.size > 0 else -5.0
            Fx_unc_l_av = np.sqrt(np.sum(xray_fluxes_unc_l**2)) / len(xray_fluxes) if xray_fluxes.size > 0 else -5.0

        else: # if I instead want the weighted average
            if radio_fluxes.size > 0:
                weights = 1 / (radio_fluxes_unc**2)
                Fr_av = np.sum(weights * radio_fluxes) / np.sum(weights)
                Fr_unc_av = 1 / np.sqrt(np.sum(weights))
            else: Fr_av, Fr_unc_av = -5.0, -5.0
            if xray_fluxes.size > 0:
                weights_u = 1 / (xray_fluxes_unc_u**2)
                weights_l = 1 / (xray_fluxes_unc_l**2)
                Fx_av = np.sum(weights_u * xray_fluxes) / np.sum(weights_u)  # Same for lower weights
                Fx_unc_u_av = 1 / np.sqrt(np.sum(weights_u))
                Fx_unc_l_av = 1 / np.sqrt(np.sum(weights_l))
            else: Fx_av, Fx_unc_u_av , Fx_unc_l_av = -5.0, -5.0, -5.0
    
        
        # Determine state (use middle value if multiple exist)
        ## TODO: Could get state using state ranges defined in obs_metadata
        xstate_used = xray_states_all[len(xray_states_all) // 2] if xray_states_all.size > 0 else None
        rstate_used = radio_states_all[len(radio_states_all) // 2] if radio_states_all.size > 0 else None
        
        midpoint_mjd = current_mjd + 0.5 * dt_mjd
        nr = len(radio_fluxes)
        nx = len(xray_fluxes)

        
        if Fx_av != -5.0 and Fr_av != -5.0:

            ## For t_diff, use the maximum difference between the radio and X-ray dates in the bin
            # Note that the code below works since the arrays are ordered
            max_dt_diff = max(
            abs(xray_MJDs_all[0] - radio_MJDs_all[-1]),
            abs(xray_MJDs_all[-1] - radio_MJDs_all[0])
            )

            paired_data.append({
                "name": source_name,
                "t": midpoint_mjd,
                "t_diff": max_dt_diff,
                "Fr": Fr_av,
                "Fr_unc": Fr_unc_av,
                "Fr_uplim_bool": radio_uplim,
                "Fx": Fx_av,
                "Fx_unc_l": Fx_unc_l_av, 
                "Fx_unc_u": Fx_unc_u_av, 
                "Fx_uplim_bool": xray_uplim,
                "state": rstate_used})
            
            print('{:<20.9f}{:<20.9f}{:<20.9f}{:<10d}{:<20.9f}{:<10d}{:<30.14f}{:<15s}{:<15s}'.format(
                current_mjd, current_mjd + dt_mjd, midpoint_mjd, nr, Fr_av, nx, Fx_av, str(radio_uplim), str(xray_uplim)))
        
        elif Fr_av != -5.0 and Fx_av == -5.0:
            unpaired_radio_dates.append(midpoint_mjd)
        
        current_mjd += dt_mjd
    
    paired_data = pd.DataFrame(paired_data)
    unpaired_radio_dates = np.array(unpaired_radio_dates).reshape(-1,)


    
    return paired_data, unpaired_radio_dates


---

# Compare Pairing for Every Source

In [3]:
def pair_comparison_runner(source_name):
    source_df, obs_df, radio_df, xray_df = read_data(f"../DATA/{source_name}.txt")

    print()
    print()

    # Current algorithm 
    print("Current algorithm (with weighted average):")
    paired_data, unpaired_radio_dates = pair_obs_alg(radio_df, xray_df, dt_mjd=1, weighted_ave = True, verbose=True)
    print(len(paired_data), "paired observations")


    print()
    print()

    # Current algorithm, but not weighted 
    print("Current algorithm, BUT without weighted average:")
    paired_data, unpaired_radio_dates = pair_obs_alg(radio_df, xray_df, dt_mjd=1, weighted_ave = False, verbose=True)
    print(len(paired_data), "paired observations")


    print()
    print()

    # Current algorith, but when there are multiple observations in a bin, take the closest to the radio instead of the average
    print("Current algorithm, but when there are multiple observations in a bin, take the closest to the radio instead of the average:")
    paired_data, unpaired_radio_dates = pair_obs_closest(radio_df, xray_df, dt_mjd=1, verbose=True)
    print(len(paired_data), "paired observations")


    print()
    print()

    # Alternative algorithm 1
    # Since it goes down the list in time, it does not always pair the closest observations within dt.
    print("Algorithm that similarly pairs within dt, going down the list in time:")
    paired_data, unpaired_radio_dates = pair_obs_closest_alt(radio_df, xray_df, dt=1)
    print(len(paired_data), "paired observations") 

    print()
    print()

    # Alternative algorithm 2
    # In this case, there are fewer observations because of the way the bins are set up
    print("Algorithm that uses fixed bins in time:")
    paired_data, unpaired_radio_dates = pair_obs_fixed_bins(radio_df, xray_df, dt_mjd=1, add_error=False, weighted_ave = False)
    print(len(paired_data), "paired observations")

In [4]:
pair_comparison_runner("1A 1744-361")

1A 1744-361
Added 5% systematic uncertainty to the radio data.
X-ray uncertainty percentage:  68
Added 10.0% systematic uncertainty to the X-ray data.


Current algorithm (with weighted average):
t_radio             Fr [mJy]            Fr_unc [mJy]        #xray     t_diff                        Mean Fx [erg/cm^2/s]          Fx_unc_l[erg/cm^2/s]          Fx_unc_u[erg/cm^2/s]          Fr_uplim_bool  Fx_uplim_bool  state          
59730.979680000     1.24000             0.06485             1         [0.765]                       2.80000e-09                   2.80247e-10                   2.80246e-10                   False          False          IMS            
59759.975490000     0.08700             0.01852             1         [0.21]                        2.79100e-09                   2.99231e-10                   2.99339e-10                   False          False          SS             
59797.938670000     0.08100             0.02700             1         [0.139]                   

In [5]:
pair_comparison_runner("4U 1543-47")

4U 1543-47
Added 5% systematic uncertainty to the radio data.
X-ray uncertainty percentage:  90
Converting uncertainties to 68% (assuming Gaussian errors).
Added 10.0% systematic uncertainty to the X-ray data.


Current algorithm (with weighted average):
t_radio             Fr [mJy]            Fr_unc [mJy]        #xray     t_diff                        Mean Fx [erg/cm^2/s]          Fx_unc_l[erg/cm^2/s]          Fx_unc_u[erg/cm^2/s]          Fr_uplim_bool  Fx_uplim_bool  state          
59392.013000000     0.25000             0.03529             2         [0.458 0.067]                 1.57154e-07                   1.11259e-08                   1.11243e-08                   False          False          SS             
59421.682200000     0.06600             0.02200             1         [0.345]                       7.10300e-08                   7.10438e-09                   7.10426e-09                   True           False          SS             
59666.058200000     6.87100          

In [6]:
pair_comparison_runner("4U 1630-47")

4U 1630-47
Added 5% systematic uncertainty to the radio data.
X-ray uncertainty percentage:  68
Added 10.0% systematic uncertainty to the X-ray data.


Current algorithm (with weighted average):
t_radio             Fr [mJy]            Fr_unc [mJy]        #xray     t_diff                        Mean Fx [erg/cm^2/s]          Fx_unc_l[erg/cm^2/s]          Fx_unc_u[erg/cm^2/s]          Fr_uplim_bool  Fx_uplim_bool  state          
58957.114000000     0.30000             0.04272             1         [0.694]                       2.02800e-08                   2.03877e-09                   2.03935e-09                   False          False          SS             
58964.171000000     0.60000             0.20000             1         [0.195]                       1.95800e-08                   1.96739e-09                   1.96787e-09                   True           False          SS             
58970.991000000     0.30000             0.10000             1         [0.872]                    

In [7]:
pair_comparison_runner("Cen X-4")

Cen X-4
Added 5% systematic uncertainty to the radio data.
X-ray uncertainty percentage:  68
Added 10.0% systematic uncertainty to the X-ray data.


Current algorithm (with weighted average):
t_radio             Fr [mJy]            Fr_unc [mJy]        #xray     t_diff                        Mean Fx [erg/cm^2/s]          Fx_unc_l[erg/cm^2/s]          Fx_unc_u[erg/cm^2/s]          Fr_uplim_bool  Fx_uplim_bool  state          
59118.623330000     0.01290             0.00430             1         [0.103]                       3.50000e-13                   1.05948e-13                   1.05948e-13                   True           False          QS             
59221.435200000     0.06900             0.02300             1         [0.285]                       7.40000e-12                   1.67260e-12                   1.67260e-12                   True           False          QS             
59230.325200000     0.04800             0.01600             1         [0.345]                       

In [8]:
pair_comparison_runner("EXO 1846-031")

EXO 1846-031
Added 5% systematic uncertainty to the radio data.
X-ray uncertainty percentage:  68
Added 10.0% systematic uncertainty to the X-ray data.


Current algorithm (with weighted average):
t_radio             Fr [mJy]            Fr_unc [mJy]        #xray     t_diff                        Mean Fx [erg/cm^2/s]          Fx_unc_l[erg/cm^2/s]          Fx_unc_u[erg/cm^2/s]          Fr_uplim_bool  Fx_uplim_bool  state          
58699.839000000     6.90000             0.36527             1         [0.571]                       1.21000e-08                   1.21413e-09                   1.21413e-09                   False          False          HS             
58705.803000000     30.80000            1.54729             1         [0.511]                       3.36000e-08                   3.36595e-09                   3.39700e-09                   False          False          IMS            
58711.904000000     12.40000            0.63789             2         [0.639 0.425]            

In [72]:
print(2.20000e-08 - 1.92743e-09)
print(1.90000e-08 + 4.42832e-09)

2.007257e-08
2.3428320000000002e-08


In [10]:
pair_comparison_runner("GRS 1739-278")

GRS 1739-278
Added 5% systematic uncertainty to the radio data.
X-ray uncertainty percentage:  68
Added 10.0% systematic uncertainty to the X-ray data.


Current algorithm (with weighted average):
t_radio             Fr [mJy]            Fr_unc [mJy]        #xray     t_diff                        Mean Fx [erg/cm^2/s]          Fx_unc_l[erg/cm^2/s]          Fx_unc_u[erg/cm^2/s]          Fr_uplim_bool  Fx_uplim_bool  state          
60132.918340000     0.18200             0.02567             1         [0.697]                       2.83790e-09                   2.86449e-10                   2.86523e-10                   False          False          SS             
60146.732920000     0.12000             0.04000             1         [0.631]                       1.06910e-09                   1.11461e-10                   1.15113e-10                   True           False          SS             
60160.832700000     0.36300             0.02701             1         [0.342]                  

In [11]:
pair_comparison_runner("GX 339-4")

GX 339-4
Added 5% systematic uncertainty to the radio data.
X-ray uncertainty percentage:  68
Added 10.0% systematic uncertainty to the X-ray data.


Current algorithm (with weighted average):
t_radio             Fr [mJy]            Fr_unc [mJy]        #xray     t_diff                        Mean Fx [erg/cm^2/s]          Fx_unc_l[erg/cm^2/s]          Fx_unc_u[erg/cm^2/s]          Fr_uplim_bool  Fx_uplim_bool  state          
58369.747590000     0.12030             0.04010             1         [0.582]                       3.02000e-13                   1.41266e-13                   1.96337e-13                   True           False          QS             
58382.699950000     0.07854             0.02618             1         [0.608]                       4.35000e-13                   1.32353e-13                   1.57141e-13                   True           False          QS             
58389.696650000     0.10590             0.03530             1         [0.205]                      

In [12]:
pair_comparison_runner("H1743-322")

H1743-322
Added 5% systematic uncertainty to the radio data.
X-ray uncertainty percentage:  68
Added 10.0% systematic uncertainty to the X-ray data.


Current algorithm (with weighted average):
t_radio             Fr [mJy]            Fr_unc [mJy]        #xray     t_diff                        Mean Fx [erg/cm^2/s]          Fx_unc_l[erg/cm^2/s]          Fx_unc_u[erg/cm^2/s]          Fr_uplim_bool  Fx_uplim_bool  state          
58375.705000000     2.42000             0.13506             1         [0.739]                       1.52000e-09                   1.71767e-10                   1.71767e-10                   False          False          HS             
58382.685000000     1.69000             0.09818             1         [0.13]                        1.31000e-09                   1.48529e-10                   1.48529e-10                   False          False          HS             
58396.662000000     0.32000             0.08158             1         [0.969]                     

In [13]:
pair_comparison_runner("IGR J17091-3624")

IGR J17091-3624
Added 5% systematic uncertainty to the radio data.
X-ray uncertainty percentage:  90
Converting uncertainties to 68% (assuming Gaussian errors).
Added 10.0% systematic uncertainty to the X-ray data.


Current algorithm (with weighted average):
t_radio             Fr [mJy]            Fr_unc [mJy]        #xray     t_diff                        Mean Fx [erg/cm^2/s]          Fx_unc_l[erg/cm^2/s]          Fx_unc_u[erg/cm^2/s]          Fr_uplim_bool  Fx_uplim_bool  state          
59657.093350000     3.98000             0.20000             1         [0.852]                       5.70000e-09                   5.70395e-10                   5.70395e-10                   False          False          HS             
59666.083070000     5.66700             0.28445             1         [0.219]                       5.53000e-09                   5.53428e-10                   5.53428e-10                   False          False          SS             
59673.060930000     5.15000     

In [14]:
pair_comparison_runner("MAXI J1348-630")

MAXI J1348-630
Added 5% systematic uncertainty to the radio data.
X-ray uncertainty percentage:  90
Converting uncertainties to 68% (assuming Gaussian errors).
Added 10.0% systematic uncertainty to the X-ray data.


Current algorithm (with weighted average):
t_radio             Fr [mJy]            Fr_unc [mJy]        #xray     t_diff                        Mean Fx [erg/cm^2/s]          Fx_unc_l[erg/cm^2/s]          Fx_unc_u[erg/cm^2/s]          Fr_uplim_bool  Fx_uplim_bool  state          
58512.029000000     13.70000            0.68675             2         [0.413 0.415]                 3.39204e-08                   2.41211e-09                   2.41223e-09                   False          False          HS             
58515.161000000     28.57000            1.43353             1         [0.624]                       6.25300e-08                   6.25515e-09                   6.25642e-09                   False          False          HS             
58523.219000000     485.60000    

In [15]:
pair_comparison_runner("MAXI J1631-479")

MAXI J1631-479
Added 5% systematic uncertainty to the radio data.
X-ray uncertainty percentage:  68
Added 10.0% systematic uncertainty to the X-ray data.


Current algorithm (with weighted average):
t_radio             Fr [mJy]            Fr_unc [mJy]        #xray     t_diff                        Mean Fx [erg/cm^2/s]          Fx_unc_l[erg/cm^2/s]          Fx_unc_u[erg/cm^2/s]          Fr_uplim_bool  Fx_uplim_bool  state          
58502.350380000     4.67000             0.30744             1         [0.468]                       6.40000e-08                   9.05097e-09                   9.05097e-09                   False          False          SS             
58509.350600000     6.88000             0.43731             1         [0.643]                       5.62000e-08                   7.94788e-09                   7.94788e-09                   False          False          HS             
58515.175730000     3.36000             0.22522             1         [0.72]                 

In [16]:
pair_comparison_runner("MAXI J1803-298")

MAXI J1803-298
Added 5% systematic uncertainty to the radio data.
X-ray uncertainty percentage:  90
Converting uncertainties to 68% (assuming Gaussian errors).
Added 10.0% systematic uncertainty to the X-ray data.


Current algorithm (with weighted average):
t_radio             Fr [mJy]            Fr_unc [mJy]        #xray     t_diff                        Mean Fx [erg/cm^2/s]          Fx_unc_l[erg/cm^2/s]          Fx_unc_u[erg/cm^2/s]          Fr_uplim_bool  Fx_uplim_bool  state          
59349.940000000     20.65700            1.03313             2         [0.014 0.609]                 1.49020e-08                   1.05524e-09                   1.05524e-09                   False          False          IMS            
59356.050000000     2.12500             0.10812             2         [0.802 0.468]                 9.56122e-09                   7.02344e-10                   7.02344e-10                   False          False          IMS            
59370.930000000     8.37400      

In [17]:
pair_comparison_runner("MAXI J1807+132")

MAXI J1807+132
Added 5% systematic uncertainty to the radio data.
X-ray uncertainty percentage:  68
Added 10.0% systematic uncertainty to the X-ray data.


Current algorithm (with weighted average):
t_radio             Fr [mJy]            Fr_unc [mJy]        #xray     t_diff                        Mean Fx [erg/cm^2/s]          Fx_unc_l[erg/cm^2/s]          Fx_unc_u[erg/cm^2/s]          Fr_uplim_bool  Fx_uplim_bool  state          
60141.748860000     0.13500             0.02016             1         [0.019]                       5.58000e-10                   5.59129e-11                   5.59127e-11                   False          False          IMS            
60146.752360000     0.21200             0.07067             4         [0.975 0.921 0.802 0.803]     2.35448e-10                   1.26495e-11                   1.26513e-11                   True           False          HS             
60153.815430000     0.13500             0.04500             1         [0.186]                

In [18]:
pair_comparison_runner("MAXI J1810-222")

MAXI J1810-222
Added 5% systematic uncertainty to the radio data.
X-ray uncertainty percentage:  68
Added 10.0% systematic uncertainty to the X-ray data.


Current algorithm (with weighted average):
t_radio             Fr [mJy]            Fr_unc [mJy]        #xray     t_diff                        Mean Fx [erg/cm^2/s]          Fx_unc_l[erg/cm^2/s]          Fx_unc_u[erg/cm^2/s]          Fr_uplim_bool  Fx_uplim_bool  state          
60078.968480000     0.24000             0.02332             1         [0.167]                       1.54730e-10                   1.57190e-11                   1.67042e-11                   False          False          HS             
60086.062570000     0.11000             0.02074             1         [0.465]                       1.48610e-10                   1.57369e-11                   1.57538e-11                   False          False          HS             
60106.965260000     0.11000             0.02074             1         [0.529]                

In [19]:
pair_comparison_runner("MAXI J1816-195")

MAXI J1816-195
Added 5% systematic uncertainty to the radio data.
X-ray uncertainty percentage:  68
Added 10.0% systematic uncertainty to the X-ray data.


Current algorithm (with weighted average):
t_radio             Fr [mJy]            Fr_unc [mJy]        #xray     t_diff                        Mean Fx [erg/cm^2/s]          Fx_unc_l[erg/cm^2/s]          Fx_unc_u[erg/cm^2/s]          Fr_uplim_bool  Fx_uplim_bool  state          
59738.974920000     4.20600             0.21220             1         [0.776]                       4.67000e-09                   4.67979e-10                   4.67960e-10                   False          False          HS             
1 paired observations


Current algorithm, BUT without weighted average:
t_radio             Fr [mJy]            Fr_unc [mJy]        #xray     t_diff                        Mean Fx [erg/cm^2/s]          Fx_unc_l[erg/cm^2/s]          Fx_unc_u[erg/cm^2/s]          Fr_uplim_bool  Fx_uplim_bool  state          
59738.974920000     

In [20]:
pair_comparison_runner("MAXI J1820+070")

MAXI J1820+070
Added 5% systematic uncertainty to the radio data.
X-ray uncertainty percentage:  68
Added 10.0% systematic uncertainty to the X-ray data.


Current algorithm (with weighted average):
t_radio             Fr [mJy]            Fr_unc [mJy]        #xray     t_diff                        Mean Fx [erg/cm^2/s]          Fx_unc_l[erg/cm^2/s]          Fx_unc_u[erg/cm^2/s]          Fr_uplim_bool  Fx_uplim_bool  state          
58389.745880000     3.47000             0.18012             2         [0.786 0.579]                 8.54484e-09                   6.25071e-10                   6.25070e-10                   False          False          IMS            
58396.695119065     11.80000            0.60289             1         [0.405]                       2.86970e-09                   2.87156e-10                   2.87155e-10                   False          False          HS             
58403.662276900     2.62000             0.13615             1         [0.605]                

In [21]:
pair_comparison_runner("SAX J1808.4-3658")

SAX J1808.4-3658
Added 5% systematic uncertainty to the radio data.
X-ray uncertainty percentage:  68
Added 10.0% systematic uncertainty to the X-ray data.


Current algorithm (with weighted average):
t_radio             Fr [mJy]            Fr_unc [mJy]        #xray     t_diff                        Mean Fx [erg/cm^2/s]          Fx_unc_l[erg/cm^2/s]          Fx_unc_u[erg/cm^2/s]          Fr_uplim_bool  Fx_uplim_bool  state          
58711.918190000     0.36465             0.02764             1         [0.735]                       3.67410e-10                   3.83414e-11                   3.84874e-11                   False          False          HS             
59818.664069000     0.37000             0.03191             2         [0.488 0.498]                 2.45854e-10                   1.79101e-11                   1.79377e-11                   False          False          HS             
59825.647711000     0.05050             0.01683             1         [0.385]              

In [22]:
pair_comparison_runner("SAX J1810.8-2609")

SAX J1810.8-2609
Added 5% systematic uncertainty to the radio data.
X-ray uncertainty percentage:  90
Converting uncertainties to 68% (assuming Gaussian errors).
Added 10.0% systematic uncertainty to the X-ray data.


Current algorithm (with weighted average):
t_radio             Fr [mJy]            Fr_unc [mJy]        #xray     t_diff                        Mean Fx [erg/cm^2/s]          Fx_unc_l[erg/cm^2/s]          Fx_unc_u[erg/cm^2/s]          Fr_uplim_bool  Fx_uplim_bool  state          
59378.005200000     0.05000             0.01619             1         [0.881]                       2.80000e-10                   2.96415e-11                   3.34162e-11                   False          False          HS             
59384.957500000     0.09100             0.03033             1         [0.83]                        2.60000e-10                   2.82091e-11                   2.87026e-11                   True           False          HS             
59392.079400000     0.10800    

In [23]:
pair_comparison_runner("Swift J1727.8-1613")

Swift J1727.8-1613
Added 5% systematic uncertainty to the radio data.
X-ray uncertainty percentage:  68
Added 10.0% systematic uncertainty to the X-ray data.


Current algorithm (with weighted average):
t_radio             Fr [mJy]            Fr_unc [mJy]        #xray     t_diff                        Mean Fx [erg/cm^2/s]          Fx_unc_l[erg/cm^2/s]          Fx_unc_u[erg/cm^2/s]          Fr_uplim_bool  Fx_uplim_bool  state          
60191.668910000     91.58000            4.57917             1         [0.364]                       2.27200e-07                   2.27576e-08                   2.27574e-08                   False          False          HS             
60193.640010000     95.57000            4.77876             3         [0.829 0.327 0.755]           2.39139e-07                   1.38317e-08                   1.38316e-08                   False          False          HS             
60195.634010000     86.64000            4.33218             1         [0.481]            

In [25]:
pair_comparison_runner("Swift J1842.5-1124")

Swift J1842.5-1124
Added 5% systematic uncertainty to the radio data.
X-ray uncertainty percentage:  68
Added 10.0% systematic uncertainty to the X-ray data.


Current algorithm (with weighted average):
t_radio             Fr [mJy]            Fr_unc [mJy]        #xray     t_diff                        Mean Fx [erg/cm^2/s]          Fx_unc_l[erg/cm^2/s]          Fx_unc_u[erg/cm^2/s]          Fr_uplim_bool  Fx_uplim_bool  state          
59001.129200000     0.06900             0.02300             1         [0.628]                       8.40400e-10                   8.41171e-11                   8.41171e-11                   True           False          SS             
1 paired observations


Current algorithm, BUT without weighted average:
t_radio             Fr [mJy]            Fr_unc [mJy]        #xray     t_diff                        Mean Fx [erg/cm^2/s]          Fx_unc_l[erg/cm^2/s]          Fx_unc_u[erg/cm^2/s]          Fr_uplim_bool  Fx_uplim_bool  state          
59001.129200000 

In [27]:
pair_comparison_runner("XTE J1701-462")

XTE J1701-462
Added 5% systematic uncertainty to the radio data.
X-ray uncertainty percentage:  68
Added 20.0% systematic uncertainty to the X-ray data.


Current algorithm (with weighted average):
t_radio             Fr [mJy]            Fr_unc [mJy]        #xray     t_diff                        Mean Fx [erg/cm^2/s]          Fx_unc_l[erg/cm^2/s]          Fx_unc_u[erg/cm^2/s]          Fr_uplim_bool  Fx_uplim_bool  state          
59829.654610000     0.07400             0.02467             3         [0.818 0.153 0.786]           7.44918e-09                   8.77092e-10                   8.77694e-10                   True           False          SS             
59838.715660000     0.33900             0.02622             1         [0.932]                       1.71000e-08                   3.42804e-09                   3.42826e-09                   False          False          SS             
59842.592190000     2.62000             0.13439             1         [0.975]                 

In [28]:
pair_comparison_runner("Vela X-1")

Vela X-1
Added 5% systematic uncertainty to the radio data.
X-ray uncertainty percentage:  68
Added 10.0% systematic uncertainty to the X-ray data.


Current algorithm (with weighted average):
t_radio             Fr [mJy]            Fr_unc [mJy]        #xray     t_diff                        Mean Fx [erg/cm^2/s]          Fx_unc_l[erg/cm^2/s]          Fx_unc_u[erg/cm^2/s]          Fr_uplim_bool  Fx_uplim_bool  state          
59119.000000000     0.09600             0.04029             1         [0.43]                        2.30000e-09                   3.04795e-10                   3.04795e-10                   False          False          Unclear        
1 paired observations


Current algorithm, BUT without weighted average:
t_radio             Fr [mJy]            Fr_unc [mJy]        #xray     t_diff                        Mean Fx [erg/cm^2/s]          Fx_unc_l[erg/cm^2/s]          Fx_unc_u[erg/cm^2/s]          Fr_uplim_bool  Fx_uplim_bool  state          
59119.000000000     0.0960